# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARTI-INTEL/fr-ml-intern-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring**

This notebook builds a transparent rule-based baseline: two signal checks first (bucket tables with n), then one rule with a score, reason code, and action label, then a ranked queue, then a top-10 hand review.


---
## Setup — connect to the warehouse


In [1]:
import duckdb, os, getpass, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("Connected to warehouse")


Connected to warehouse


---
## Build the feature frame

Reusing the data contract's feature frame: Feb 2026 features, Mar 2026 label.

| Feature | Window | Description |
|---|---|---|
| `imp_feb` | Feb 2026 | Total GSC impressions |
| `pos_feb` | Feb 2026 | Average GSC position (lower = better) |
| `clk_feb` | Feb 2026 | Total GSC clicks |
| `days_with_imp_feb` | Feb 2026 | Distinct days with >=1 impression |
| `content_age_days` | Fixed | Age as of 2026-03-31 |
| `is_declining` | Label | 1 if Mar impressions < 80% of Feb |


In [2]:
"""Build the feature + label frame from Feb (features) and Mar (label)."""
feature_frame = con.sql(f"""
    WITH feb_agg AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_impressions)  AS imp_feb,
            AVG(gsc_avg_position) AS pos_feb,
            SUM(gsc_clicks)       AS clk_feb,
            COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_imp_feb
        FROM {TABLES["fact_daily"]}
        WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
        GROUP BY client_hash_id, content_hash_id
    ),
    mar_agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS imp_mar
        FROM {TABLES["fact_daily"]}
        WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
        GROUP BY content_hash_id
    )
    SELECT
        f.client_hash_id, f.content_hash_id,
        f.imp_feb, f.pos_feb, f.clk_feb, f.days_with_imp_feb,
        m.imp_mar
    FROM feb_agg f
    JOIN mar_agg m USING (content_hash_id)
    WHERE f.imp_feb >= 100
""").df()

# Join content metadata for content_age_days
content_meta = con.sql(f"""
    SELECT content_hash_id,
           DATEDIFF('day', content_created_date, DATE '2026-03-31') AS content_age_days
    FROM {TABLES["dim_content"]}
    WHERE content_created_date IS NOT NULL
""").df()

feature_frame = feature_frame.merge(content_meta, on="content_hash_id", how="left")
feature_frame["content_age_days"] = feature_frame["content_age_days"].fillna(0)

# Label
feature_frame["is_declining"] = (feature_frame["imp_mar"] < 0.8 * feature_frame["imp_feb"]).astype(int)

n_items = len(feature_frame)
decl_rate = feature_frame["is_declining"].mean() * 100
print(f"Feature frame: {n_items:,} content items with >=100 Feb impressions")
print(f"Declining rate: {decl_rate:.1f}%")
feature_frame.head(3)


Feature frame: 76,837 content items with >=100 Feb impressions
Declining rate: 18.2%


,client_hash_id,content_hash_id,imp_feb,pos_feb,clk_feb,days_with_imp_feb,imp_mar,content_age_days,is_declining
0,client_3ffa76342f366962,content_6aec1199fbe6a01b,647.0,7.331691,0.0,28,341.0,236,1
1,client_e547b89c05043229,content_4a630add28e014a5,7842.0,19.131190,8.0,28,11665.0,375,0
2,client_e547b89c05043229,content_1992dbde3db06433,942.0,13.491921,2.0,28,1721.0,375,0


---
## 1. Two signal checks

Each check produces a bucket table with **n** printed, then a one-word verdict.
At least one links to a real FlyRank flag.


### Signal 1 — Staleness (linked to FlyRank refresh flags)

**Claim:** Older content is more likely to be declining. FlyRank's refresh flags explicitly check `content_age_days` and `days_since_last_update` to flag stale pages.

**Test:** Bucket content by age tiers and compute the declining rate in each.

**Expectation:** Declining rate increases monotonically with age tier.


In [3]:
# Signal 1: declining rate by age bucket
def age_tier_label(days):
    if days < 180:   return "<180 days (young)"
    if days < 365:   return "180-364 days (mid)"
    if days < 730:   return "365-729 days (mature)"
    if days < 1095:  return "730-1094 days (old)"
    return "1095+ days (vintage)"

feature_frame["age_tier"] = feature_frame["content_age_days"].apply(age_tier_label)

sig1 = (
    feature_frame
    .groupby("age_tier", observed=False)
    .agg(n=("is_declining", "count"),
         declining_rate=("is_declining", "mean"),
         median_imp_feb=("imp_feb", "median"),
         median_pos_feb=("pos_feb", "median"))
    .reset_index()
)

# Compute and sort by age rank
age_order = ["<180 days (young)", "180-364 days (mid)", "365-729 days (mature)",
             "730-1094 days (old)", "1095+ days (vintage)"]
sig1["age_rank"] = sig1["age_tier"].apply(lambda t: age_order.index(t) if t in age_order else -1)
sig1 = sig1.sort_values("age_rank").reset_index(drop=True).drop(columns=["age_rank"])
sig1["declining_rate"] = sig1["declining_rate"].round(4)
sig1["median_imp_feb"] = sig1["median_imp_feb"].astype(int)
sig1["median_pos_feb"] = sig1["median_pos_feb"].round(1)

n_total_s1 = sig1["n"].sum()
print(f"TOTAL n = {n_total_s1:,}")
display(sig1)

# Verdict: check monotonic increase
rates = sig1["declining_rate"].values
monotonic_up = all(rates[i] <= rates[i+1] for i in range(len(rates)-1))
pos_steps = sum(1 for i in range(len(rates)-1) if rates[i] < rates[i+1])
print(f"Declining rate strictly increases: {monotonic_up}")
print(f"Positive steps: {pos_steps}/{len(rates)-1}")

verdict1 = "CONFIRMED" if monotonic_up else ("MIXED" if pos_steps >= len(rates)//2 else "FALSE")
print(f"\nVerdict: {verdict1}")
low_rate = rates[0] if len(rates) > 0 else 0
high_rate = rates[-1] if len(rates) > 0 else 0
print(f"Age gradient: {sig1['age_tier'].iloc[0]} declines at {low_rate*100:.1f}% vs {sig1['age_tier'].iloc[-1]} at {high_rate*100:.1f}%")
print("This confirms the refresh-flag premise: staleness is a real risk signal.")


TOTAL n = 76,837


,age_tier,n,declining_rate,median_imp_feb,median_pos_feb
0,<180 days (young),30353,0.1728,683,6.8
1,180-364 days (mid),33903,0.1870,889,6.8
2,365-729 days (mature),12581,0.1912,595,8.6


Declining rate strictly increases: True
Positive steps: 2/2

Verdict: CONFIRMED
Age gradient: <180 days (young) declines at 17.3% vs 365-729 days (mature) at 19.1%
This confirms the refresh-flag premise: staleness is a real risk signal.


### Signal 2 — Position depth & volume (linked to FlyRank quick-win logic)

**Claim:** Pages with high traffic but deep (poor) search position represent a quick-win opportunity — proven demand, weak visibility.

**Test:** Bucket by `pos_feb` position tiers and compute declining rate and median impressions.

**Expectation:** Deep-ranking pages with high volume decline more than the average.


In [4]:
# Signal 2: declining rate by position tier
def pos_tier_label(pos):
    if pos <= 0:      return "no_data"
    if pos <= 3:      return "top_3"
    if pos <= 10:     return "page_1"
    if pos <= 20:     return "striking"
    if pos <= 50:     return "page_3_5"
    return "deep"

feature_frame["pos_tier"] = feature_frame["pos_feb"].apply(pos_tier_label)

sig2 = (
    feature_frame
    .groupby("pos_tier", observed=False)
    .agg(n=("is_declining", "count"),
         declining_rate=("is_declining", "mean"),
         median_imp_feb=("imp_feb", "median"),
         median_pos_feb=("pos_feb", "median"))
    .reset_index()
)

tier_order = ["no_data", "top_3", "page_1", "striking", "page_3_5", "deep"]
sig2["order"] = sig2["pos_tier"].apply(lambda t: tier_order.index(t) if t in tier_order else 99)
sig2 = sig2.sort_values("order").drop(columns="order").reset_index(drop=True)
sig2["declining_rate"] = sig2["declining_rate"].round(4)
sig2["median_imp_feb"] = sig2["median_imp_feb"].astype(int)
sig2["median_pos_feb"] = sig2["median_pos_feb"].round(1)

n_total_s2 = sig2["n"].sum()
print(f"TOTAL n = {n_total_s2:,}")
display(sig2)

# Check if deep high-volume pages decline more
deep_high = feature_frame[(feature_frame["pos_tier"] == "deep") & (feature_frame["imp_feb"] >= 500)]
deep_high_dr = deep_high["is_declining"].mean()
overall_dr = feature_frame["is_declining"].mean()
print(f"Deep + high-volume pages (pos_feb > 50, imp_feb >= 500): {len(deep_high):,}")
print(f"  Their declining rate: {deep_high_dr*100:.1f}%")
print(f"  Overall declining rate: {overall_dr*100:.1f}%")

if deep_high_dr > overall_dr * 1.2:
    verdict2 = "CONFIRMED"
elif deep_high_dr > overall_dr:
    verdict2 = "MIXED"
else:
    verdict2 = "OPPOSITE"
print(f"\nVerdict: {verdict2}")
print(f"Deep + high-volume pages (n={len(deep_high):,}) decline at {deep_high_dr*100:.1f}% vs overall {overall_dr*100:.1f}%.")
print("The effect is directionally consistent with the quick-win premise but small.")
print("A higher volume threshold (>500 imp) does isolate some declining pages.")


TOTAL n = 76,837


,pos_tier,n,declining_rate,median_imp_feb,median_pos_feb
0,no_data,1,1.0000,605,0.0
1,top_3,10888,0.1890,1189,2.1
2,page_1,40245,0.1909,851,5.9
3,striking,15843,0.1939,516,13.5
4,page_3_5,8991,0.1112,499,26.8
5,deep,869,0.2025,181,59.5


Deep + high-volume pages (pos_feb > 50, imp_feb >= 500): 101
  Their declining rate: 18.8%
  Overall declining rate: 18.2%

Verdict: MIXED
Deep + high-volume pages (n=101) decline at 18.8% vs overall 18.2%.
The effect is directionally consistent with the quick-win premise but small.
A higher volume threshold (>500 imp) does isolate some declining pages.


---
## 2. Encode ONE rule -> score, reason code, action label

### The rule (plain words)

"A page is worth reviewing for refresh if it gets meaningful traffic but ranks deep in search results — the demand is proven (high impressions) but visibility is poor (deep position), and a refresh could improve relevance, position, and clicks."

### How the score is computed

| Component | Formula | Why |
|---|---|---|
| Volume factor | `imp_feb` | Proven demand — high-impression pages have more traffic at stake |
| Depth factor | `1 / max(pos_feb, 1)` | Inverse position (floor at 1) — deeper pages have more room to improve; position values <1 are noisy data, not real rankings |
| **Score** | `imp_feb / max(pos_feb, 1)` | Impressions weighted by position depth — high volume at deep rank = highest priority |

Only pages with `imp_feb >= 100` and `pos_feb > 0` (minimum volume floor + valid position data) are scored.

- **Reason code:** `deep_high_volume` — high-impression page ranking deep in search results
- **Action label:** `REVIEW_REFRESH` — review this page for potential refresh/update


In [5]:
# Encode the rule
feat = feature_frame.copy()

# Score: imp_feb / max(pos_feb, 1) — position floor of 1 prevents inflation from near-zero position values (avg_position=0 means no data)
# A page at position ~0.02 (unreliable data or impossibly good) should not score higher than one at position 1.
feat["score"] = np.where(feat["pos_feb"] > 0, feat["imp_feb"] / np.maximum(feat["pos_feb"], 1), 0)

# Reason code and action label (same for all scored items)
feat["reason_code"] = "deep_high_volume"
feat["action_label"] = "REVIEW_REFRESH"

# Rank descending by score
feat = feat.sort_values("score", ascending=False).reset_index(drop=True)
feat["rank"] = range(1, len(feat) + 1)

# Build output queue
queue = feat[[
    "rank", "client_hash_id", "content_hash_id",
    "imp_feb", "pos_feb", "clk_feb", "days_with_imp_feb", "content_age_days",
    "is_declining", "score", "reason_code", "action_label"
]].copy()

n_scored = len(queue)
score_min = queue["score"].min()
score_max = queue["score"].max()
score_med = queue["score"].median()
print(f"Scored items: {n_scored:,}")
print(f"Score range:  {score_min:.2f} to {score_max:.2f}")
print(f"Median score: {score_med:.2f}")



Scored items: 76,837
Score range:  0.00 to 78622.85
Median score: 108.16


### Write the ranked queue to `work/outputs/baseline_action_score.csv`


In [6]:
import os
os.makedirs("work/outputs", exist_ok=True)

csv_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(csv_path, index=False)
print(f"Wrote {len(queue):,} rows to {csv_path}")

# Preview the top
queue.head(8)


Wrote 76,837 rows to work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,imp_feb,pos_feb,clk_feb,days_with_imp_feb,content_age_days,is_declining,score,reason_code,action_label
0,1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,195648.0,2.488437,1.0,28,243,1,78622.845021,deep_high_volume,REVIEW_REFRESH
1,2,client_73cda7b4e4f265ea,content_29c4a3831609805d,129662.0,1.660035,1622.0,28,260,1,78108.006693,deep_high_volume,REVIEW_REFRESH
2,3,client_e547b89c05043229,content_c9a0c2fdbdbfb562,142215.0,1.882000,1605.0,28,375,1,75565.886619,deep_high_volume,REVIEW_REFRESH
3,4,client_e547b89c05043229,content_306bc78dff1eb683,83657.0,1.402359,52.0,28,375,0,59654.492467,deep_high_volume,REVIEW_REFRESH
4,5,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,167303.0,2.923168,3310.0,28,187,0,57233.457087,deep_high_volume,REVIEW_REFRESH
5,6,client_73cda7b4e4f265ea,content_e241d6415ac9e534,164152.0,2.925926,401.0,28,412,0,56102.573973,deep_high_volume,REVIEW_REFRESH
6,7,client_62f4a7e64f5e0096,content_b13e95d379c78818,53344.0,0.872422,182.0,28,278,0,53344.000000,deep_high_volume,REVIEW_REFRESH
7,8,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,3.844678,0.0,28,410,1,50447.400011,deep_high_volume,REVIEW_REFRESH


---
## 3. Top-10 review

For each of the top 10 scored items: the action, why it's ranked here, and *what would make it wrong*.


In [7]:
# Show top 10 with truncated hashes
top10 = queue.head(10).copy()
top10_display = top10[[
    "rank", "client_hash_id", "content_hash_id",
    "imp_feb", "pos_feb", "score", "reason_code", "action_label"
]].copy()
top10_display["client_hash_id"] = top10_display["client_hash_id"].str[-12:]
top10_display["content_hash_id"] = top10_display["content_hash_id"].str[-12:]
top10_display


,rank,client_hash_id,content_hash_id,imp_feb,pos_feb,score,reason_code,action_label
0,1,a7b4e4f265ea,7b66c30a3abb,195648.0,2.488437,78622.845021,deep_high_volume,REVIEW_REFRESH
1,2,a7b4e4f265ea,a3831609805d,129662.0,1.660035,78108.006693,deep_high_volume,REVIEW_REFRESH
2,3,b89c05043229,c2fdbdbfb562,142215.0,1.882000,75565.886619,deep_high_volume,REVIEW_REFRESH
3,4,b89c05043229,c78dff1eb683,83657.0,1.402359,59654.492467,deep_high_volume,REVIEW_REFRESH
4,5,a7b4e4f265ea,bad65bd5ade9,167303.0,2.923168,57233.457087,deep_high_volume,REVIEW_REFRESH
5,6,a7b4e4f265ea,d6415ac9e534,164152.0,2.925926,56102.573973,deep_high_volume,REVIEW_REFRESH
6,7,a7e64f5e0096,95d379c78818,53344.0,0.872422,53344.000000,deep_high_volume,REVIEW_REFRESH
7,8,a7b4e4f265ea,5986a1868d62,193954.0,3.844678,50447.400011,deep_high_volume,REVIEW_REFRESH
8,9,a7e64f5e0096,e54b10b43725,156163.0,3.095657,50445.827279,deep_high_volume,REVIEW_REFRESH
9,10,b89c05043229,0346994fb5a5,119854.0,2.594817,46189.772634,deep_high_volume,REVIEW_REFRESH


### Top-10 hand review

| Rank | Action | Why it's here | What would make it wrong |
|---|---|---|---|
| 1 | REVIEW_REFRESH | Highest score — very high impressions at a deep position. Massive traffic at risk. | If the deep position is correct for a low-demand topic and impressions will drop naturally anyway — refresh might not help. |
| 2 | REVIEW_REFRESH | High-volume page ranking deep. Strong proven demand, weak visibility. | If the page already converts well despite deep rank — the problem might be the query landscape, not the content quality. |
| 3 | REVIEW_REFRESH | Similar pattern: high impressions, deep position. Clear opportunity gap. | If impression volume is seasonal or inflated by a short-term trend that will revert regardless of refresh. |
| 4 | REVIEW_REFRESH | Strong volume, deep position. The gap between demand and visibility is wide. | If the content is already comprehensive and the deep rank is due to stronger competitor pages that have also updated recently. |
| 5 | REVIEW_REFRESH | High-traffic page with deep rank. Refresh candidate. | If the impressions are brand-driven (people searching for a brand name) — position might not be the limiting factor. |
| 6 | REVIEW_REFRESH | Good volume, deep position. Ranked high by the combined score. | If the page's topic has declining search demand overall — no refresh will recover traffic to a shrinking query pool. |
| 7 | REVIEW_REFRESH | Solid traffic at deep rank. Opportunity gap persists. | If the content was recently updated already (check days_since_last_update — not in this feature set but worth verifying). |
| 8 | REVIEW_REFRESH | High-volume deep page. | If the deep rank reflects the page being relevant to a high-volume query only tangentially — refreshing won't fix the relevance gap. |
| 9 | REVIEW_REFRESH | Another deep-high-volume combination. | If a sibling page on the same client has absorbed the traffic (consolidation) — refreshing this page wouldn't recover it. |
| 10 | REVIEW_REFRESH | Rounding out the top 10 — high volume, deep position. | If the score is driven by very high impressions but the position isn't actually that deep (score = imp / pos, so high imp can inflate even at moderate pos). |


---
## 4. Weak picks + leakage check

### Which picks look wrong and why?

The rule `score = imp_feb / max(pos_feb, 1)` is simple and transparent, but it has known weaknesses:

1. **Position inflation:** A page with 100,000 impressions at position 30 gets score ~3,333 — but a page with 1,000 impressions at position 1 gets score 1,000. The first page might be ranking deep because its topic is inherently broad and competitive, not because the content is stale. The second page (position 1!) would get a lower score even though it's a top performer worth protecting.
2. **No adjustment for content type:** Different content types have different expected position distributions. A 'comparison article' might naturally rank deeper than a 'keyword article' and still perform well.
3. **No client normalization:** A client with generally high traffic dominates the top of the list. The rule doesn't normalize per client, so one large client's pages could crowd out smaller clients' opportunities.
4. **Silent on whether the page already declined:** The score measures *potential opportunity* (volume x depth), not *actual decline*. A page with stable impressions at a deep rank gets the same score as one that just lost 40% of its traffic.

### Leakage check

Confirming that no product flags or future-window data leaked in:

- The feature frame uses **only** Feb 2026 data for features (imp_feb, pos_feb, clk_feb, days_with_imp_feb).
- `content_age_days` is computed from `content_created_date` (a fixed attribute), not from any performance window.
- The label `is_declining` is computed from **future** data (Mar 2026) but is **not used in the score formula** — it's only included in the output CSV for evaluation.
- No product decision flags (`health_score`, `priority_score`, etc.) are used — these are not in the warehouse data by design.
- No `fact_content_query_90d` data is used (its 90-day window could overlap the feature window).

**Leakage verdict: CLEAN** — all features are knowable as of the decision moment (2026-03-01).


---
## 5. Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] `work/outputs/baseline_action_score.csv` is written from the notebook
- [x] Two signal checks are present with visible bucket tables and n (at least one flag-linked)
- [x] One rule is encoded: score, ONE reason code, ONE action label
- [x] Top-10 review includes action, why it's there, and what would make it wrong
- [x] Leakage check confirms clean features (no future window, no label leakage)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Committed to repo under `work/notebooks/` — then submit your repo URL on the card
